In [2]:
import requests
import json

base_url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.term": "pediatric cancer",
    "filter.overallStatus": "TERMINATED,WITHDRAWN,UNKNOWN,SUSPENDED,NO_LONGER_AVAILABLE",
    "fields": "NCTId,BriefTitle,OverallStatus,WhyStopped,StartDate,CompletionDate,LocationCountry,LeadSponsorName",
    "pageSize": 200
}

all_studies = []
while True:
    r = requests.get(base_url, params=params)
    data = r.json()
    all_studies.extend(data["studies"])
    print(f"Fetched {len(all_studies)} so far...")
    token = data.get("nextPageToken")
    if not token:
        break
    params["pageToken"] = token

with open("all_stalled_trials.json", "w") as f:
    json.dump({"studies": all_studies}, f)

print(f"Done. Total: {len(all_studies)}")

Fetched 200 so far...
Fetched 400 so far...
Fetched 600 so far...
Fetched 800 so far...
Fetched 950 so far...
Done. Total: 950


In [ ]:
print(f"Shape of the flattened DataFrame: {df_flattened.shape}")
print("Columns in the flattened DataFrame:")
for col in df_flattened.columns:
    print(f"- {col}")

print("\nValue counts for OverallStatus:")
display(df_flattened['OverallStatus'].value_counts())

print("\nValue counts for WhyStopped (top 10):")
display(df_flattened['WhyStopped'].value_counts().head(10))

In [4]:
def extract_protocol_info(protocol_section):
    nct_id = protocol_section.get('identificationModule', {}).get('nctId')
    brief_title = protocol_section.get('identificationModule', {}).get('briefTitle')
    overall_status = protocol_section.get('statusModule', {}).get('overallStatus')
    why_stopped = protocol_section.get('statusModule', {}).get('whyStopped')
    start_date = protocol_section.get('statusModule', {}).get('startDateStruct', {}).get('date')
    completion_date = protocol_section.get('statusModule', {}).get('primaryCompletionDateStruct', {}).get('date') \
                    or protocol_section.get('statusModule', {}).get('completionDateStruct', {}).get('date')

    location_country_list = protocol_section.get('locationModule', {}).get('locations')
    location_country = None
    if location_country_list:
        # Assuming one country for simplicity, or taking the first one found
        location_country = location_country_list[0].get('country')

    lead_sponsor_name = protocol_section.get('sponsorCollaboratorsModule', {}).get('leadSponsor', {}).get('leadSponsorName')

    return {
        'NCTId': nct_id,
        'BriefTitle': brief_title,
        'OverallStatus': overall_status,
        'WhyStopped': why_stopped,
        'StartDate': start_date,
        'CompletionDate': completion_date,
        'LocationCountry': location_country,
        'LeadSponsorName': lead_sponsor_name
    }

In [5]:
# Apply the function to the 'protocolSection' column
flattened_data = df['protocolSection'].apply(extract_protocol_info)

# Convert the list of dictionaries into a DataFrame
flattened_df = pd.DataFrame(flattened_data.tolist())

# Drop the original 'protocolSection' column and concatenate with the new flattened DataFrame
df_flattened = pd.concat([df.drop(columns=['protocolSection']), flattened_df], axis=1)

display(df_flattened.head())

,NCTId,BriefTitle,OverallStatus,WhyStopped,StartDate,CompletionDate,LocationCountry,LeadSponsorName
0,NCT00679536,Allogeneic Transplantation for Pediatric Leuke...,UNKNOWN,None,2008-05,2015-05,None,None
1,NCT01757899,Effects and Safety of Infusion of Low-Doses of...,WITHDRAWN,Technical problems,2014-01,2016-01,None,None
2,NCT05689892,Shared Decision Making in Paediatric Inflammat...,UNKNOWN,None,2024-01,2025-12,None,None
3,NCT05179057,Posoleucel (ALVR105) for the Treatment of Aden...,TERMINATED,Study discontinued as DSMB determined it was f...,2022-04-26,2024-01-31,None,None
4,NCT04288700,Evaluation of the Efficacy of Captopril Versus...,UNKNOWN,None,2019-10-01,2021-10-01,None,None


In [3]:
import pandas as pd
import json

with open("all_stalled_trials.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data["studies"])
display(df.head())

,protocolSection
0,{'identificationModule': {'nctId': 'NCT0067953...
1,{'identificationModule': {'nctId': 'NCT0175789...
2,{'identificationModule': {'nctId': 'NCT0568989...
3,{'identificationModule': {'nctId': 'NCT0517905...
4,{'identificationModule': {'nctId': 'NCT0428870...


In [6]:
print(df_flattened.isnull().sum())
print(f"\)nTotal rows: {len(df_flattened)}")

NCTId                0
BriefTitle           0
OverallStatus        0
WhyStopped         512
StartDate           15
CompletionDate      60
LocationCountry    950
LeadSponsorName    950
dtype: int64
\)nTotal rows: 950


<>:2: SyntaxWarning: invalid escape sequence '\)'
<>:2: SyntaxWarning: invalid escape sequence '\)'
/tmp/ipykernel_14284/3814657822.py:2: SyntaxWarning: invalid escape sequence '\)'
  print(f"\)nTotal rows: {len(df_flattened)}")


In [8]:
rows = []

for study in data["studies"]:
    proto = study["protocolSection"]

    ident = proto.get("identificationModule", {})
    status = proto.get("statusModule", {})
    sponsor = proto.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})
    locations = proto.get("contactsLocationsModule", {}).get("locations", [])

    countries = list(set(l["country"] for l in locations if "country" in l))

    rows.append({
        "NCTId": ident.get("nctId"),
        "BriefTitle": ident.get("briefTitle"),
        "OverallStatus": status.get("overallStatus"),
        "WhyStopped": status.get("whyStopped"),
        "StartDate": status.get("startDateStruct", {}).get("date"),
        "CompletionDate": status.get("completionDateStruct", {}).get("date"),
        "LeadSponsorName": sponsor.get("name"),
        "LocationCountries": "|".join(countries),
        "NumCountries": len(countries)
    })

df_clean = pd.DataFrame(rows)

print(df_clean.isnull().sum())
print(df_clean.head())

NCTId                  0
BriefTitle             0
OverallStatus          0
WhyStopped           512
StartDate             15
CompletionDate        60
LeadSponsorName        0
LocationCountries      0
NumCountries           0
dtype: int64
         NCTId                                         BriefTitle  \
0  NCT00679536  Allogeneic Transplantation for Pediatric Leuke...   
1  NCT01757899  Effects and Safety of Infusion of Low-Doses of...   
2  NCT05689892  Shared Decision Making in Paediatric Inflammat...   
3  NCT05179057  Posoleucel (ALVR105) for the Treatment of Aden...   
4  NCT04288700  Evaluation of the Efficacy of Captopril Versus...   

  OverallStatus                                         WhyStopped  \
0       UNKNOWN                                               None   
1     WITHDRAWN                                 Technical problems   
2       UNKNOWN                                               None   
3    TERMINATED  Study discontinued as DSMB determined it was f... 

In [9]:
df_clean.to_csv("stalled_trials_clean.csv", index=False)

from google.colab import files
files.download("stalled_trials_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>